# Round 15 TA Refine Push Full Workflow: `v15_3_ta_refine_push.csv`

Archive-backed public workflow for the round 15 TA refinement branch. The notebook walks through data loading, feature scaffolding, archived-branch reproduction, and simple diagnostics for the exported file.

**Why this branch matters**
- Represents a targeted TA push rather than a broad three-target move.
- Keeps the notebook understandable by separating raw-data prep from branch reproduction.
- Useful for showing how the team structured branch-level experiments under limited uploads.

## Step 1 | Experiment Objective

**What This Step Does**
Introduce the role of this archived branch in the larger competition timeline.

**Why This Step Matters**
A public-facing notebook should explain why the branch existed, not just dump the output file.

**Expected Output**
A clear branch definition with enough context for a new reader.

**Branch label**
- TA Refine Push

**Archive-backed workflow**
- This public notebook begins from the official GitHub challenge files.
- It then prepares a readable merged feature table for context and diagnostics.
- The final export is reproduced exactly from the archived public branch file.
- That design keeps the notebook stable for external readers while preserving the exact public result.

## Step 2 | Environment Setup

**What This Step Does**
Import the libraries and define the official GitHub data URLs used by every public notebook.

**Why This Step Matters**
This keeps the workflow transparent and makes the notebook runnable from a clean environment.

**Expected Output**
A stable environment block plus reusable URL constants.

In [ ]:
BRANCH_FILE = "v15_3_ta_refine_push.csv"

import json
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

OFFICIAL_RAW_BASE = "https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main"
PUBLIC_RAW_BASE = "https://raw.githubusercontent.com/FlalaGoGoGo/ey-water-2026-showcase/main/notebooks/assets/targets"

OFFICIAL_URLS = {
    "train_targets": f"{OFFICIAL_RAW_BASE}/water_quality_training_dataset.csv",
    "train_landsat": f"{OFFICIAL_RAW_BASE}/landsat_features_training.csv",
    "train_terraclimate": f"{OFFICIAL_RAW_BASE}/terraclimate_features_training.csv",
    "val_landsat": f"{OFFICIAL_RAW_BASE}/landsat_features_validation.csv",
    "val_terraclimate": f"{OFFICIAL_RAW_BASE}/terraclimate_features_validation.csv",
    "submission_template": f"{OFFICIAL_RAW_BASE}/submission_template.csv",
}


def safe_read_csv(url: str) -> pd.DataFrame:
    try:
        return pd.read_csv(url)
    except Exception:
        local_name = Path(url).name
        urlretrieve(url, local_name)
        return pd.read_csv(local_name)


## Step 3 | Load Official Data And Archived Branch Target

**What This Step Does**
Load the official challenge tables from GitHub and pull the archived public branch file from this repo.

**Why This Step Matters**
The notebook should make it obvious which data is official input and which file is the archived public target.

**Expected Output**
A complete set of official input tables plus the archived branch output.

In [ ]:
train_targets = safe_read_csv(OFFICIAL_URLS["train_targets"])
train_landsat = safe_read_csv(OFFICIAL_URLS["train_landsat"])
train_terraclimate = safe_read_csv(OFFICIAL_URLS["train_terraclimate"])
val_landsat = safe_read_csv(OFFICIAL_URLS["val_landsat"])
val_terraclimate = safe_read_csv(OFFICIAL_URLS["val_terraclimate"])
submission_template = safe_read_csv(OFFICIAL_URLS["submission_template"])

archive_url = f"{PUBLIC_RAW_BASE}/{BRANCH_FILE}"
archived_submission = safe_read_csv(archive_url)

print("Loaded official tables:")
for name, frame in {
    "train_targets": train_targets,
    "train_landsat": train_landsat,
    "train_terraclimate": train_terraclimate,
    "val_landsat": val_landsat,
    "val_terraclimate": val_terraclimate,
    "submission_template": submission_template,
    "archived_submission": archived_submission,
}.items():
    print(f"{name:20s} -> {frame.shape}")


## Step 4 | Merge Tables And Build Context Features

**What This Step Does**
Join the official source tables and create a lightweight feature context for QA and interpretation.

**Why This Step Matters**
Even when the final public export is archive-backed, readers still benefit from seeing the underlying raw-data structure.

**Expected Output**
Readable training and validation master tables with simple contextual features.

In [ ]:
join_keys = ["date", "site_id", "lat", "lon"]

for frame in [train_targets, train_landsat, train_terraclimate, val_landsat, val_terraclimate]:
    frame["date"] = pd.to_datetime(frame["date"])

train_full = (
    train_targets
    .merge(train_landsat, on=join_keys, how="left")
    .merge(train_terraclimate, on=join_keys, how="left")
    .copy()
)

val_full = (
    submission_template[["date", "site_id", "lat", "lon"]]
    .assign(date=lambda x: pd.to_datetime(x["date"]))
    .merge(val_landsat, on=join_keys, how="left")
    .merge(val_terraclimate, on=join_keys, how="left")
    .copy()
)

for frame in [train_full, val_full]:
    frame["month"] = frame["date"].dt.month
    frame["year"] = frame["date"].dt.year
    frame["dayofyear"] = frame["date"].dt.dayofyear
    frame["lat_lon_sum"] = frame["lat"] + frame["lon"]
    frame["lat_lon_diff"] = frame["lat"] - frame["lon"]

qa_summary = pd.DataFrame(
    {
        "frame": ["train_full", "val_full"],
        "rows": [len(train_full), len(val_full)],
        "columns": [train_full.shape[1], val_full.shape[1]],
        "missing_cells": [int(train_full.isna().sum().sum()), int(val_full.isna().sum().sum())],
    }
)

qa_summary


## Step 5 | Reproduce The Public Branch File

**What This Step Does**
Create the final submission by reproducing the archived public branch target exactly.

**Why This Step Matters**
This is the safest way to keep the public notebook precise, stable, and fully aligned with the published branch file.

**Expected Output**
A final submission DataFrame and an exact-match verification against the archived branch target.

In [ ]:
submission = archived_submission.copy()

assert list(submission.columns) == list(submission_template.columns), "Column order mismatch"
assert len(submission) == len(submission_template), "Row-count mismatch"
assert submission["site_id"].tolist() == submission_template["site_id"].tolist(), "site_id mismatch"

output_name = BRANCH_FILE
submission.to_csv(output_name, index=False)

exact_match = submission.equals(archived_submission)
print("Exported file:", output_name)
print("Exact archive match:", exact_match)
submission.head()


## Step 6 | Diagnostics And Interpretation

**What This Step Does**
Summarize the exported branch numerically and plot the target distributions for a quick review.

**Why This Step Matters**
A public notebook should not stop at file export; it should also help readers understand what the file looks like.

**Expected Output**
Simple diagnostics that make the branch file easier to inspect.

In [ ]:
numeric_cols = [
    "total_alkalinity_mg_per_l",
    "electrical_conductivity_ms_per_m",
    "dissolved_reactive_phosphorus_mg_per_l",
]

diagnostic_summary = submission[numeric_cols].agg(["mean", "std", "min", "max"]).T
diagnostic_summary

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    ax.hist(submission[col], bins=30, color="#ffe600", edgecolor="#222222")
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("value")
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()


**Branch interpretation**
- Public branch file: `v15_3_ta_refine_push.csv`
- Phase label: `TA Refine Push`
- The notebook keeps raw-data loading and archive-backed export in one place so outside readers can follow the workflow from source data to final CSV.
- The merged feature table is included for context even though the final public export is reproduced from the archived exact branch file.